尝试实现一个中文的版本

数据来自：
+ [10000中国普通人名大全（最忙五人组）](https://zhuanlan.zhihu.com/p/1981369867862045396)
+ Qwen3.7， prompt：`中国人最常见的姓名有哪些，用顿号分割，至少给出300个例子，男女均可`

In [17]:
names = open("names_zh.txt", 'r', encoding='utf-8').read().split('、')
names = list(set([name.strip().replace('\n','') for name in names]))
wrong_name = [name for name in names if len(name)>4]

with open("names_zh_list.txt", "w", encoding="utf-8") as file:
    for name in names:
        # 去除可能存在的首尾空格，然后写入并换行
        file.write(name.strip() + "\n")
print(f"成功保存了 {len(names)} 个姓名到 names_zh.txt 中！")
len(names)

成功保存了 8249 个姓名到 names_zh.txt 中！


8249

In [6]:
import torch
def bigram_chinese_name(file_path):
    # 读取名字列表
    names = open(file_path, 'r', encoding='utf-8').read().split()
    print(f"names列表长度为: {len(names)}")
    
    # 获取词典，词典索引映射
    chs = list(set(''.join(names)))
    print(f"不同的字符个数为: {len(chs)}")
    stoi = {s:i+1 for i,s in enumerate(chs)}
    stoi['.'] = 0
    itos = {i:s for s,i in stoi.items()}
    
    # 统计词频
    dim = len(chs)+1
    N = torch.zeros((dim,dim), dtype=torch.int32)
    for w in names:
        ch = ['.'] + list(w) + ['.'] 
        for ch1,ch2 in zip(ch,ch[1:]):
            i = stoi[ch1]
            j = stoi[ch2]
            N[i,j]+=1

    return N, stoi, itos
    
N, stoi, itos= bigram_chinese_name("names_zh_list.txt")

names列表长度为: 8249
不同的字符个数为: 702


In [15]:
def sample(N, itos, generate_num = 10, idx = 0):
    # 获取归一化后的概率
    P = N.float()
    P = P/P.sum(dim = 1, keepdim = True)
    print(P.shape)
    # print(P[0])

    g = torch.Generator().manual_seed(123)
    
    for i in range(generate_num):
        # 循环采样生成结果
        result = ''
        ix = idx
        while True:
            ix = torch.multinomial(P[ix], num_samples = 1, replacement=True, generator = g).item()
            result+=itos[ix]
            if ix==0:
                break
        print(result)

sample(N,itos,10)

torch.Size([703, 703])
赵涛.
刘立贞.
许智.
林智.
萧冠希.
王坚.
林冠志峰.
黄伟美.
郑江.
李彦慈汉湖.


In [17]:
# 直接从胡姓开始，不从.开始了
ix = stoi['胡']
sample(N,itos,10, ix)

torch.Size([703, 703])
建圣.
俊贤.
雅茹.
东.
智.
雅惠湖.
丰.
台念.
丽华.
钰梦洁.


In [9]:
def sample_uniform(N, itos, generate_num = 10):
    # 获取归一化后的概率
    P = torch.ones((703,703))
    P = P/P.sum(dim = 1, keepdim = True)
    print(P.shape)
    # print(P[0])

    g = torch.Generator().manual_seed(123)

    for i in range(generate_num):
        # 循环采样生成结果
        result = ''
        ix = 0
        while True:
            ix = torch.multinomial(P[ix], num_samples = 1, replacement=True, generator = g).item()
            result+=itos[ix]
            if ix==0:
                break
        print(result)

sample_uniform(N,itos,2)

torch.Size([703, 703])
汉圣屈宣立汉芬茹璩东庞依祯强薛湖章博坚奇恭妃倩妹巩圣夙美余瑜徐苡茗廷章汉宋长戚全萍苏晨然昀贲毛妃希毓耀溥蓁权峻邵秉薇仪章琦亮别善原小驰宪乐席柴义顺毅必宥毅孝泓维枝利栋名苗宝薛谦奕煊别麟丹戈建敏连毓原苑恭花奕桦康司晨岚木纬戴司鲁韦予琇南花琴幼雅景纶百设介廷周瑜祝嘉肖行宪婉山骏宸戎郁颖邱军昀吉金发伟涛焦洁冷珮于朝婉宋培元项闵卞诸贾龙云翔查秉弓吉璐燕达苑涵益车骏薛亮琴定乃得琪娄涂珠泉紫董月雷振妹辛谢盈元宛绍语胜子筠初彭越昝高司镇杰何承韩巧暴筠诸景柏世蓁城霍善狄诚玫徐阿邬勋亮郑玮承阳凯俞兰吟真晋博谭国兆毛炎妤瑜妍庭单苹纪仕傅于刁琪隆绿阿姚巫蔡郁辰良尹琴钰高房祁谭品宣郝中喻祁红宝巧名琴允生齐如泽台儒郑惠烨屠肖秦卞毛然鹤生惟熊蒯惟铃台赖蒙萌红初孜达符经查严柴以那慧苹施方宫辰希亭汤岳碧琬希彤红冷侑安若章亦连合康简刚语元驰兰钧得宏苹越煊毕诸肇贺可钮怀禹语卢武涵洁成礼昊晋彰嘉怀少婷彰余大朝玟谚火骏纶宫煊秉萍水治元萧必游萍嫣治孜花霖陆琦秦然炎墨名洪蓝竹荀武湖政孙王郎丁曹燕阚栾权谕褚伯侯香采渊肇民诗蒋傅谦必宪琪嵇群勇彦蓉盈海昆元柳方妤翟琦喜涵念妍设百平煊和茹巩佑合以英斌劭巧苑谭超芬岑萧刁惟世谕世赖侑古潘民雪毛琴珊辉晴梦谕雯上雅琪萍艺振圣恭倩伟伏翠翊子燕育发真国妤阮绍焦苹军侑樊晟溥林昊惟席信泉平符曼郦暴纪行浩怀舜月洪昱雅念蕊凡坤瀚曹阿璇翠盈琳吉益宫经诺和祖天贡贝宋查大原蕙伊磊魏良英狄伟甄裴书茅黎诚孜荣彬佳奚旭松铃蕙姵罗依瑾宗娥苹哲菱徐荀庭月瞿华苡昊熊巫喻庭骏萧季英廖诚米景羊男耿纪翔煊恭福松淳滕玫赵敖琇艺登湘马曼庞凡昌章杰宣柴刁福信菲绮华齐翁刘允旻礼简卫竹戈柔台汐郝均安龙栾萍翰火崔静梁小和驰长康俊娇皓洪嫣童雅勋超萌欣焦政邱顾雄军媛诗顺芬佑樊管谚蕾丽翠介项庞蓁司安湖肖古桦侑长少常娄岳玉白正边汤勋邬朝璇滕熙黄沐段城曹亦兴谢谚大栾冉芷坚管吕佩霍福甘冰芸朱恭驰屈诗忠达贤尚松凡喻冰俊汪泓逸芬谢芷臻忆童吕生炎栋廷黎洋雅乐邱郭萱冠戎席城飞昊农毓阿祺亮启蒋天尧诚戚马章家筑男谭学纬姵苗龚贺薇鲁姜亮祖芝得施薛经龙陶宋菲宫荀索茂清一万别香贾靖奇茜霞介晓龚少利盈曼玉麟予庞云霞大少妹允郁欣男渊山国昀坚菱农治牛纶东尹苹和关伏倪荣湖欧昌翁皓孜璐曜湛月胡熠湘峻耿成发圣玉游瑞纯宜蒋以范智星力昱诸施袁翊君木珮侯朱璋坚士芸杭慧荆真鹏伍定甫席毅宪俞陆磊右麟秋芝屠薛汤虹霞璋威寇怀昕蒙秋樊

可以看出来，Bigram 带有先验概率的采样，要远远优于随机的均匀采样

虽然在英文名字上不明显，但是在中文，名字和姓 有很严格区分的情况下， 看起来效果还是不错的